# SET Pharma — Google Colab Pipeline

This notebook runs the existing `SET_new` Python implementation. It does not duplicate model, feature-engineering, reliability or RADCSR logic.

**Research boundary:** Inventory outcomes are simulated. Stockout labels are not observed events.

Run the cells in order. The notebook checkpoints ingestion, the model pipeline and mathematical validation separately. The current `run_model_pipeline()` is one monolithic stage; an interruption inside it requires rerunning that stage.

## 00. Runtime diagnostics

Inspect the allocated runtime before copying or processing the dataset. A GPU is optional; this workload may be limited by RAM, CPU and disk rather than GPU.

In [ ]:
import os, sys, shutil, subprocess, json, time
from pathlib import Path

print("Python:", sys.version)
if sys.version_info[:2] != (3, 13):
    raise RuntimeError(
        f"This Colab notebook requires Python 3.13.x; detected {sys.version.split()[0]}. "
        "Select a Python 3.13 Colab runtime and restart the session."
    )
print("Python 3.13 runtime verified.")
print("CPU cores:", os.cpu_count())
print("Disk free (GB):", round(shutil.disk_usage('/content').free / 1024**3, 2))
try:
    import psutil
    print("RAM total (GB):", round(psutil.virtual_memory().total / 1024**3, 2))
    print("RAM available (GB):", round(psutil.virtual_memory().available / 1024**3, 2))
except ImportError:
    print("psutil not installed yet")
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch is not installed; GPU diagnostics skipped")

## 01. Clone the feature branch

The notebook deliberately checks out `feature/colab-pipeline-execution`, not `main`.

In [ ]:
REPO_URL = "https://github.com/Souptik-Hazra/SET_new.git"
BRANCH = "feature/colab-pipeline-execution"
DEFAULT_NOTEBOOK_COMMIT = "1a93d377c121898dee5585eb45997e9f84cdeb43"
NOTEBOOK_COMMIT = DEFAULT_NOTEBOOK_COMMIT  # Override only after reviewing the replacement commit.
if NOTEBOOK_COMMIT != DEFAULT_NOTEBOOK_COMMIT:
    print(f"WARNING: NOTEBOOK_COMMIT override active: {NOTEBOOK_COMMIT}")
REPO_DIR = Path("/content/SET_new")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)

# Execute the reviewed snapshot, not the moving branch head.
subprocess.run(["git", "checkout", NOTEBOOK_COMMIT], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Repository:", REPO_DIR)
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO_DIR, text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True).strip())

## 02. Install project dependencies

Use the repository's pinned dependency ranges. Installation is performed in the active Colab runtime.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "psutil"], check=True)
print("Dependencies installed. If Colab requests a runtime restart, restart and continue from section 00.")

## 03. Mount Google Drive and configure paths

Place the source CSV at the path below in Drive, or edit `DRIVE_RAW_CSV` to match its actual location. The raw file is copied to Colab's local disk for processing; it is never uploaded to Git.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_RAW_CSV = Path("/content/drive/MyDrive/Dataset/combined_secondary_care_medicines.csv")
DRIVE_ARTIFACT_ROOT = Path("/content/drive/MyDrive/SET_Pharma_Colab_Artifacts")
LOCAL_RAW_CSV = REPO_DIR / "data" / "raw" / "combined_secondary_care_medicines.csv"

if not DRIVE_RAW_CSV.is_file():
    raise FileNotFoundError(f"Upload the SCMD CSV to {DRIVE_RAW_CSV}, or update DRIVE_RAW_CSV.")
LOCAL_RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
required_bytes = DRIVE_RAW_CSV.stat().st_size
free_bytes = shutil.disk_usage("/content").free
if free_bytes < required_bytes * 1.15:
    raise RuntimeError(f"Insufficient Colab disk space: need approximately {required_bytes * 1.15 / 1024**3:.2f} GB; available {free_bytes / 1024**3:.2f} GB")
if (not LOCAL_RAW_CSV.is_file()
    or LOCAL_RAW_CSV.stat().st_size != required_bytes
    or LOCAL_RAW_CSV.stat().st_mtime_ns != DRIVE_RAW_CSV.stat().st_mtime_ns):
    print("Copying raw CSV from Drive to local Colab storage...")
    shutil.copy2(DRIVE_RAW_CSV, LOCAL_RAW_CSV)
print("Raw CSV:", LOCAL_RAW_CSV, "| GB:", round(LOCAL_RAW_CSV.stat().st_size / 1024**3, 3))
print("Persistent artifacts:", DRIVE_ARTIFACT_ROOT)

## 04. Execution controls

Keep the research defaults unless a documented experiment requires a change. Set `FORCE_* = True` only when intentionally rebuilding a stage. A checkpoint is reused only when its fingerprint matches and all expected outputs exist.

In [ ]:
from src.set_forecasting.config import get_config, validate_config

CONFIG = validate_config(get_config())
RUN_INGESTION = True
RUN_MODEL_PIPELINE = True
RUN_MATHEMATICAL_VALIDATION = True

FORCE_INGESTION = False
FORCE_MODEL_PIPELINE = False
FORCE_MATHEMATICAL_VALIDATION = False

MIN_PERIODS = CONFIG["data"]["min_periods"]
MIN_NONZERO = CONFIG["data"]["min_nonzero"]
TRAIN_FRAC = CONFIG["data"]["train_frac"]
VALIDATION_FRAC = CONFIG["data"]["validation_frac"]
TEST_FRAC = CONFIG["data"]["test_frac"]
BOOTSTRAP_RESAMPLES = CONFIG["evaluation"]["bootstrap_resamples"]
RANDOM_SEED = CONFIG["seed"]

print(json.dumps({k: v for k, v in globals().copy().items() if k in [
    "RUN_INGESTION", "RUN_MODEL_PIPELINE", "RUN_MATHEMATICAL_VALIDATION",
    "FORCE_INGESTION", "FORCE_MODEL_PIPELINE", "FORCE_MATHEMATICAL_VALIDATION",
    "MIN_PERIODS", "MIN_NONZERO", "TRAIN_FRAC", "VALIDATION_FRAC", "TEST_FRAC", "BOOTSTRAP_RESAMPLES", "RANDOM_SEED"
]}, indent=2))


## 05. Restore previous checkpoints and import the runner

Only processed Parquet files and reports are restored from Drive. The raw CSV and repository are not copied back from the artifact folder.

In [ ]:
from scripts.colab_runtime import (
    environment_report, sync_artifacts, run_ingestion,
    run_models, run_mathematical_validation, DISCLAIMER
)

print(json.dumps(environment_report(), indent=2))
restored = sync_artifacts(DRIVE_ARTIFACT_ROOT, REPO_DIR) if DRIVE_ARTIFACT_ROOT.exists() else []
print("Restored artifact files:", len(restored))
print(DISCLAIMER)

## 06. Phase 1 — SCMD ingestion

DuckDB streams and aggregates the CSV. The monthly cohort is written to Parquet. The raw file remains outside Git. After success, processed outputs and the stage manifest are copied to Drive.

In [ ]:
if RUN_INGESTION:
    ingestion_result = run_ingestion(
        LOCAL_RAW_CSV,
        min_periods=MIN_PERIODS,
        min_nonzero=MIN_NONZERO,
        train_frac=TRAIN_FRAC,
        val_frac=VALIDATION_FRAC,
        test_frac=TEST_FRAC,
        force=FORCE_INGESTION,
    )
    print(ingestion_result)
    saved = sync_artifacts(REPO_DIR, DRIVE_ARTIFACT_ROOT)
    print("Persisted files to Drive:", len(saved))
else:
    print("Ingestion disabled by configuration")

## 07. Forecasting, reliability, RADCSR and simulation

This invokes the canonical `run_model_pipeline()` from the existing Python package. It is currently a single long-running stage. If Colab disconnects while this function is running, rerun this cell; the completed ingestion checkpoint will be reused. Model-stage internal checkpoints are a future refactor, not simulated here.

In [ ]:
if RUN_MODEL_PIPELINE:
    model_result = run_models(
        LOCAL_RAW_CSV,
        min_periods=MIN_PERIODS,
        min_nonzero=MIN_NONZERO,
        train_frac=TRAIN_FRAC,
        val_frac=VALIDATION_FRAC,
        test_frac=TEST_FRAC,
        bootstrap=BOOTSTRAP_RESAMPLES,
        seed=RANDOM_SEED,
        force=FORCE_MODEL_PIPELINE,
    )
    print(model_result)
    saved = sync_artifacts(REPO_DIR, DRIVE_ARTIFACT_ROOT)
    print("Persisted files to Drive:", len(saved))
else:
    print("Model pipeline disabled by configuration")

## 08. Phase B mathematical validation

This requires the forecast and simulation artifacts from the model stage. It evaluates the current implementation and writes JSON and Markdown reports.

In [ ]:
if RUN_MATHEMATICAL_VALIDATION:
    math_result = run_mathematical_validation(
        bootstrap=BOOTSTRAP_RESAMPLES,
        seed=RANDOM_SEED,
        force=FORCE_MATHEMATICAL_VALIDATION,
    )
    print("Mathematical validation status:", math_result["status"])
    saved = sync_artifacts(REPO_DIR, DRIVE_ARTIFACT_ROOT)
    print("Persisted files to Drive:", len(saved))
else:
    print("Mathematical validation disabled by configuration")

## 09. Generate, save and display research figures

Figures are generated from the pipeline's saved Parquet and CSV artifacts. The generator does not insert illustrative metric values. Figures based on synthetic inventory outcomes are explicitly labelled as simulated. Each PNG is saved under `output/figures/`, displayed below, and synchronized to Google Drive.

In [ ]:
from IPython.display import Image, display
from src.visualization.generate_figures import generate_figures

generated_figures = generate_figures(REPO_DIR)
if generated_figures:
    for figure_path in generated_figures:
        print(figure_path.name)
        display(Image(filename=str(figure_path), width=1000))
else:
    print("No figures were generated. Inspect output/figures/figure_generation_report.json for missing inputs.")

saved = sync_artifacts(REPO_DIR, DRIVE_ARTIFACT_ROOT)
print("Persisted figures and reports to Drive:", len(saved))
print(DISCLAIMER)

## 10. Review generated outputs

Inspect reports before drawing research conclusions. Simulated stockout labels are not observed hospital stockout events.

In [ ]:
report_paths = [
    REPO_DIR / "output" / "pipeline_report.json",
    REPO_DIR / "output" / "mathematical_validation.json",
    REPO_DIR / "output" / "final_verdict.md",
    REPO_DIR / "output" / "figures" / "figure_generation_report.json",
]
for path in report_paths:
    print(("FOUND  " if path.is_file() else "MISSING ") + str(path))
    if path.is_file(): print("  Size:", path.stat().st_size, "bytes")

processed = REPO_DIR / "data" / "processed"
print("\nProcessed artifacts:")
for path in sorted(processed.glob("*")):
    if path.is_file(): print(f"{path.name:45s} {path.stat().st_size / 1024**2:8.2f} MB")

figures_dir = REPO_DIR / "output" / "figures"
print("\nSaved figures:")
for path in sorted(figures_dir.glob("*.png")):
    print(f"{path.name:48s} {path.stat().st_size / 1024:.1f} KB")

print("\n", DISCLAIMER)

## 11. Re-running selected stages

- To rerun only mathematical validation, set `RUN_INGESTION = False`, `RUN_MODEL_PIPELINE = False`, `RUN_MATHEMATICAL_VALIDATION = True` and rerun the relevant cells.
- To rebuild forecasts and downstream outputs from an existing monthly series, disable ingestion and force the model stage.
- To rebuild the monthly cohort, force ingestion; downstream stages should then be rerun.
- If the runtime disconnects, reconnect, rerun setup and Drive restore cells, then rerun the desired stage cells.

The checkpoint fingerprint includes source-file metadata, hashes of the code files used by that stage, the configuration hash and stage parameters. The Git revision is recorded in the manifest for audit. Changing relevant implementation code or configuration invalidates the affected checkpoint.